# Part 1: 데이터 검증 및 전처리

> CSV 검증 → Parquet 통합 → 25개 파생변수 생성

**데이터 소스**: BigQuery `daiso` 데이터셋

In [1]:
import os
import warnings
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from IPython.display import display
from google.cloud import bigquery
from google.oauth2 import service_account

mpl.rcParams["font.family"] = "Malgun Gothic"
mpl.rcParams["axes.unicode_minus"] = False
warnings.filterwarnings("ignore")

PROJECT_ROOT = Path(r"G:/Final_proj/Total_clear/데이터")
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT

SERVICE_KEY_NAME = "daiso-analysis-4d05c813a295.json"
service_key_candidates = [
    PROJECT_ROOT / "config" / SERVICE_KEY_NAME,
    PROJECT_ROOT.parent / "config" / SERVICE_KEY_NAME,
]
env_key = os.environ.get("GOOGLE_APPLICATION_CREDENTIALS")
if env_key:
    service_key_candidates.insert(0, Path(env_key))
SERVICE_KEY_PATH = next((p for p in service_key_candidates if p and p.exists()), None)


def get_client():
    if SERVICE_KEY_PATH:
        credentials = service_account.Credentials.from_service_account_file(str(SERVICE_KEY_PATH))
        return bigquery.Client(credentials=credentials, project=credentials.project_id)
    return bigquery.Client()


def query_to_df(sql: str) -> pd.DataFrame:
    return get_client().query(sql).to_dataframe()


def list_tables(dataset: str = "daiso") -> list:
    client = get_client()
    tables = client.list_tables(f"{client.project}.{dataset}")
    return [t.table_id for t in tables]


print(f"프로젝트 루트: {PROJECT_ROOT}")
if SERVICE_KEY_PATH:
    print(f"BigQuery 키 파일: {SERVICE_KEY_PATH}")
else:
    print("BigQuery 키 파일을 찾지 못했습니다. 기본 인증(ADC)으로 연결을 시도합니다.")


BigQuery 연결 완료
프로젝트 루트: /Users/yu_seok/Documents/Document/workspace/03_프로젝트/04_Why-pi


## 1.1 데이터 현황 검증 (BigQuery)

In [2]:
# BigQuery 테이블 현황
tables = list_tables()
print(f'BigQuery 테이블: {len(tables)}개')
print('=' * 50)
for t in sorted(tables):
    try:
        cnt = query_to_df(f"SELECT COUNT(*) AS cnt FROM daiso.{t}").iloc[0, 0]
        print(f'  {t:30s} {cnt:>10,}행')
    except:
        print(f'  {t:30s} (조회 실패)')

BigQuery 테이블: 52개
  brands                                 92행
  functional                            256행
  ingredients_dic                     1,741행
  manufacturer                            0행
  pipeline_log                            1행
  products_category                     950행
  products_core                         950행
  products_ingredients               27,975행
  products_stats                        950행
  promotions                             16행
  review_absa                       393,229행
  review_aspects                    479,616행
  reviews_core                      393,239행
  reviews_text                      393,239행
  search_trends                      11,920행
  sli_results                           949행
  user_id_map                        26,800행
  users_profile                      26,800행
  users_repurchase                   14,806행
  v_brand_alert_status                  947행
  v_brand_benchmark                      86행
  v_brand_dashboard_master           

In [3]:
# 제품-브랜드-카테고리 매칭 검증
df_check = query_to_df("""
SELECT
    (SELECT COUNT(*) FROM daiso.products_core)                              AS total_products,
    (SELECT COUNT(DISTINCT brand_id) FROM daiso.products_core)              AS unique_brands,
    (SELECT COUNT(*) FROM daiso.reviews_core)                               AS total_reviews,
    (SELECT COUNT(DISTINCT product_code) FROM daiso.reviews_core)           AS reviewed_products,
    (SELECT COUNT(DISTINCT product_code) FROM daiso.products_ingredients)   AS products_with_ingredients
""")
print('데이터 현황:')
for col in df_check.columns:
    print(f'  {col}: {df_check[col].iloc[0]:,}')

데이터 현황:
  total_products: 950
  unique_brands: 90
  total_reviews: 393,239
  reviewed_products: 1,017
  products_with_ingredients: 915


# Feature Engineering
다이소 뷰티 제품 데이터 파생변수 생성

In [4]:
# 제품 데이터 (BigQuery)
products_df = query_to_df("""
SELECT
    pc.product_code,
    b.name  AS brand,
    pc.name,
    pc.price,
    pc.country,
    pcat.category_1,
    pcat.category_2,
    ps.likes,
    ps.shares,
    ps.review_count,
    ps.first_review_date,
    ps.engagement_score,
    ps.cp_index,
    ps.review_density,
    ps.risk_score,
    CASE
        WHEN pc.price <= 1000 THEN 'Easy Pick'
        WHEN pc.price <= 3000 THEN 'Standard'
        ELSE 'Premium'
    END AS price_tier,
    f.is_whitening,
    f.is_wrinkle_reduction,
    f.is_sunscreen,
    f.is_acne
FROM daiso.products_core   pc
LEFT JOIN daiso.brands            b    ON pc.brand_id     = b.brand_id
LEFT JOIN daiso.products_category pcat ON pc.product_code = pcat.product_code
LEFT JOIN daiso.products_stats    ps   ON pc.product_code = ps.product_code
LEFT JOIN daiso.functional        f    ON pc.product_code = f.product_code
""")

# is_god_sung_bi (cp_index 상위 20%)
q80 = products_df['cp_index'].quantile(0.8)
products_df['is_god_sung_bi'] = products_df['cp_index'] >= q80

print(f"제품 데이터: {len(products_df):,}개")
print(f"컬럼: {products_df.columns.tolist()}")

제품 데이터: 950개
컬럼: ['product_code', 'brand', 'name', 'price', 'country', 'category_1', 'category_2', 'likes', 'shares', 'review_count', 'first_review_date', 'engagement_score', 'cp_index', 'review_density', 'risk_score', 'price_tier', 'is_whitening', 'is_wrinkle_reduction', 'is_sunscreen', 'is_acne', 'is_god_sung_bi']


In [5]:
# 리뷰 데이터 (BigQuery)
reviews_df = query_to_df("""
SELECT
    rc.review_id,
    rc.product_code,
    CAST(rc.user_id AS STRING) AS user,
    rc.user_id,
    rc.rating,
    rc.review_date,
    rc.image_count,
    rc.is_reorder,
    rc.promotion_id,
    rt.text,
    rt.review_length
FROM daiso.reviews_core  rc
LEFT JOIN daiso.reviews_text rt ON rc.review_id = rt.review_id
""")

# 유저 프로필 (별도 로드 — 스키마 호환)
_up = query_to_df("SELECT * FROM daiso.users_profile")
reviews_df = reviews_df.merge(_up, on='user_id', how='left')

# 유저 재구매 (별도 로드)
_ur = query_to_df("SELECT * FROM daiso.users_repurchase")
reviews_df = reviews_df.merge(_ur, on='user_id', how='left')

# user_masked (user_id_map 에서 로드)
_um = query_to_df("SELECT user_id, user_masked FROM daiso.user_id_map")
reviews_df = reviews_df.merge(_um, on='user_id', how='left')

# 파생변수 추가
reviews_df['review_date'] = pd.to_datetime(reviews_df['review_date'])
reviews_df['write_date']  = reviews_df['review_date']
reviews_df['date']        = reviews_df['review_date']

# 시즌
month = reviews_df['review_date'].dt.month
reviews_df['season'] = np.where(month.isin([3,4,5]), 'Spring',
                      np.where(month.isin([6,7,8]), 'Summer',
                      np.where(month.isin([9,10,11]), 'Fall', 'Winter')))

print(f"리뷰 데이터: {len(reviews_df):,}개")
print(f"컬럼: {reviews_df.columns.tolist()}")

리뷰 데이터: 393,239개
컬럼: ['review_id', 'product_code', 'user', 'user_id', 'rating', 'review_date', 'image_count', 'is_reorder', 'promotion_id', 'text', 'review_length', 'user_total_reviews', 'user_activity_level', 'user_rating_tendency', 'review_tenure', 'reorder_user_category', 'reorder_user_brand', 'reorder_user_avg_rating', 'user_masked', 'write_date', 'date', 'season']


## A. 상품 관점: "진짜 인기 상품" 찾기

In [6]:
# 제품별 리뷰 수 계산
review_counts = reviews_df.groupby('product_code').size().reset_index(name='review_count_calc')

# BQ에서 이미 review_count가 있으면 drop 후 재병합
if 'review_count' in products_df.columns:
    products_df = products_df.drop(columns=['review_count'])

products_df = products_df.merge(review_counts, on='product_code', how='left')
products_df = products_df.rename(columns={'review_count_calc': 'review_count'})
products_df['review_count'] = products_df['review_count'].fillna(0).astype(int)

print(f"리뷰 수 추가 완료")
print(f"리뷰가 있는 제품: {(products_df['review_count'] > 0).sum()}개")
print(f"리뷰가 없는 제품: {(products_df['review_count'] == 0).sum()}개")

리뷰 수 추가 완료
리뷰가 있는 제품: 949개
리뷰가 없는 제품: 1개


### 가중치 결정을 위한 데이터 분석

In [7]:
# 상관관계와 분별력 분석
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler

print("="*80)
print("Engagement Score 가중치 결정을 위한 데이터 분석")
print("="*80)

# 1. 기본 통계량
print("\n[1] 기본 통계량")
print("-"*80)
stats = products_df[['likes', 'shares', 'review_count']].describe()
print(stats)

Engagement Score 가중치 결정을 위한 데이터 분석

[1] 기본 통계량
--------------------------------------------------------------------------------
             likes     shares  review_count
count        950.0      950.0    950.000000
mean   2445.157895  39.164211    379.650526
std    2647.682125  73.884628    739.490294
min            0.0        0.0      0.000000
25%         691.25        8.0     85.000000
50%         1390.5       18.0    170.500000
75%         3150.5      41.75    373.000000
max         9999.0     1096.0   9789.000000


In [8]:
# 2. 변동계수 (Coefficient of Variation) = std / mean
# 값이 클수록 분산이 크고 분별력이 높음
print("\n변동계수 - 분별력 지표")
print("-"*80)
cv_likes = products_df['likes'].std() / products_df['likes'].mean()
cv_shares = products_df['shares'].std() / products_df['shares'].mean()
cv_reviews = products_df['review_count'].std() / products_df['review_count'].mean()

print(f"likes:        CV = {cv_likes:.3f}")
print(f"shares:       CV = {cv_shares:.3f}")
print(f"review_count: CV = {cv_reviews:.3f}")
print(f"\n분별력 순위: ", end="")
cv_dict = {'likes': cv_likes, 'shares': cv_shares, 'review_count': cv_reviews}
sorted_cv = sorted(cv_dict.items(), key=lambda x: x[1], reverse=True)
print(" > ".join([f"{k} ({v:.3f})" for k, v in sorted_cv]))


변동계수 - 분별력 지표
--------------------------------------------------------------------------------
likes:        CV = 1.083
shares:       CV = 1.887
review_count: CV = 1.948

분별력 순위: review_count (1.948) > shares (1.887) > likes (1.083)


In [9]:
# 3. 정규화 후 분산
# 값이 클수록 분별력이 낮음
print("\n정규화 후 분산")
print("-"*80)
scaler = MinMaxScaler()
normalized = scaler.fit_transform(products_df[['likes', 'shares', 'review_count']])
normalized_df = pd.DataFrame(normalized, columns=['likes_norm', 'shares_norm', 'reviews_norm'])

norm_var_likes = normalized_df['likes_norm'].var()
norm_var_shares = normalized_df['shares_norm'].var()
norm_var_reviews = normalized_df['reviews_norm'].var()

print(f"likes:        {norm_var_likes:.4f}")
print(f"shares:       {norm_var_shares:.4f}")
print(f"review_count: {norm_var_reviews:.4f}")


정규화 후 분산
--------------------------------------------------------------------------------
likes:        0.0701
shares:       0.0045
review_count: 0.0057


In [10]:
# 4. 상관관계 분석
print("\n상관관계 매트릭스")
print("-"*80)
corr = products_df[['likes', 'shares', 'review_count']].corr()
print(corr)


상관관계 매트릭스
--------------------------------------------------------------------------------
                 likes    shares  review_count
likes         1.000000  0.662635      0.687040
shares        0.662635  1.000000      0.704345
review_count  0.687040  0.704345      1.000000


In [11]:
# A-1. Engagement Score (실질적 인기도)
# Score = w1*likes + w2*shares + w3*review_count
# 가중치: review_count(0.55) > shares(0.30) > likes(0.15)
# 분별력이 높은 변수에 높은 가중치 부여
w1, w2, w3 = 0.15, 0.30, 0.55  # likes, shares, review_count

products_df['engagement_score'] = (
    w1 * products_df['likes'] + 
    w2 * products_df['shares'] + 
    w3 * products_df['review_count']
)

print("Engagement Score 생성 완료")
print(f"가중치: likes={w1}, shares={w2}, review_count={w3}")
print(f"평균: {products_df['engagement_score'].mean():.2f}")
print(f"최대: {products_df['engagement_score'].max():.2f}")
print("\nTOP 5 인기 제품:")
print(products_df.nlargest(5, 'engagement_score')[['name', 'brand', 'likes', 'shares', 'review_count', 'engagement_score']])

Engagement Score 생성 완료
가중치: likes=0.15, shares=0.3, review_count=0.55
평균: 587.33
최대: 6984.00

TOP 5 인기 제품:
                                  name  brand  likes  shares  review_count  \
296  VT 리들샷 100 페이셜 부스팅 퍼스트 앰플 2ml*6개입     VT   9999     334          9789   
293  VT 리들샷 300 페이셜 부스팅 퍼스트 앰플 2ml*6개입     VT   9999     209          9784   
714             본셉 비타씨 동결 건조 더블샷 앰플 키트     본셉   9999    1096          6357   
586      마데카21 테카 솔루션 수딩 미스트 토너 200 ml  마데카21   9999     890          6203   
717       본셉 레티놀 2500 IU 링클샷 퍼펙터 15 ml     본셉   9999     416          4980   

     engagement_score  
296            6984.0  
293           6943.75  
714            5325.0  
586            5178.5  
717           4363.65  


In [12]:
# A-2-1. price_tier (절대적 가격 티어)
def classify_daiso_price(price):
    if price <= 1500:
        return 'Easy Pick (Low)'
    elif price <= 3000:
        return 'Standard (Mid)'
    else: # 5000원 등
        return 'Premium (High)'

products_df['price_tier'] = products_df['price'].apply(classify_daiso_price)

print("price_tier 생성 완료")
print("\n가격대별 분포:")
print(products_df['price_tier'].value_counts())
print("\n카테고리별 가격대 분포:")
print(pd.crosstab(products_df['category_2'], products_df['price_tier']))

price_tier 생성 완료

가격대별 분포:
price_tier
Standard (Mid)     464
Premium (High)     429
Easy Pick (Low)     57
Name: count, dtype: int64

카테고리별 가격대 분포:
price_tier  Easy Pick (Low)  Premium (High)  Standard (Mid)
category_2                                                 
기초스킨케어                    6             176              42
남성메이크업                    0               2               0
남성스킨케어                    1              18               7
남성용면도기                    6               8              16
남성향수                      0               0               6
립메이크업                     2              24             106
립케어                       4               6              19
베이스메이크업                   0              69              35
아이메이크업                   12              26              90
자외선차단제                    2              34               5
치크/하이라이터                  0              16              65
클렌징/쉐이빙                   0               2               7
클렌징/필링      

In [13]:
# A-2-2. price_position (카테고리 대비 가격 지수)
# 1. 소분류별 평균 가격 계산
category_avg_price = products_df.groupby('category_2')['price'].transform('mean')

# 2. 비율 계산 (1.0 = 평균, 1.5 = 평균보다 1.5배 비쌈)
products_df['relative_price_ratio'] = products_df['price'] / category_avg_price

# 3. 구간화
def categorize_relative_price(ratio):
    if ratio < 0.8: return 'Cheaper than Avg'
    elif ratio > 1.2: return 'More Expensive than Avg'
    else: return 'Average'

products_df['price_position'] = products_df['relative_price_ratio'].apply(categorize_relative_price)

print("Price Position 생성완료\n")
print(products_df['price_position'].value_counts(normalize=True).mul(100).round(1))
print("\n카테고리별 Price Position 분포")
print(pd.crosstab(products_df['category_2'],products_df['price_position'],normalize='index').mul(100).round(1).head(5))


Price Position 생성완료

price_position
Average                    65.1
Cheaper than Avg           20.6
More Expensive than Avg    14.3
Name: proportion, dtype: float64

카테고리별 Price Position 분포
price_position  Average  Cheaper than Avg  More Expensive than Avg
category_2                                                        
기초스킨케어             78.6              21.4                      0.0
남성메이크업            100.0               0.0                      0.0
남성스킨케어             69.2              30.8                      0.0
남성용면도기             30.0              43.3                     26.7
남성향수              100.0               0.0                      0.0


In [14]:
# A-2-3. price_tier (가성비 지수)
# 숫자가 너무 작아지는 것을 방지하기 위해 * 1000
products_df['cp_index'] = (products_df['engagement_score'] / products_df['price']) * 1000

# 상위 10% 
quantile_90 = products_df['cp_index'].quantile(0.9)
products_df['is_god_sung_bi'] = products_df['cp_index'] >= quantile_90

print("Price_tier 생성 완료")
print("\n가성비(Top 10%) 선정 상품의 가격대 분포")
god_sung_bi_items = products_df[products_df['is_god_sung_bi'] == True]
print(god_sung_bi_items['price'].value_counts(normalize=True).sort_index().mul(100).round(1))
print("\n전체 상품 가격대 분포")
print(products_df['price'].value_counts(normalize=True).sort_index().mul(100).round(1))
print("\n가성비 아이템 예시 (Top 5)")
display(god_sung_bi_items[['category_1', 'name', 'price', 'cp_index']].sort_values(by='cp_index', ascending=False).head(5))

Price_tier 생성 완료

가성비(Top 10%) 선정 상품의 가격대 분포
price
500.0      6.3
1000.0     9.5
2000.0     8.4
3000.0    46.3
5000.0    29.5
Name: proportion, dtype: float64

전체 상품 가격대 분포
price
500.0      0.9
1000.0     5.1
2000.0     6.6
3000.0    42.2
5000.0    45.2
Name: proportion, dtype: float64

가성비 아이템 예시 (Top 5)


,category_1,name,price,cp_index
4,스킨케어,에그캡슐팩,500.0,7012.8
44,스킨케어,팩미인 하이드로겔 브이 마스크팩,1000.0,3123.7
5,스킨케어,감자 캡슐팩,500.0,2569.9
48,스킨케어,VT PDRN 광채시트마스크 1매 25 g,1000.0,2433.1
296,스킨케어,VT 리들샷 100 페이셜 부스팅 퍼스트 앰플 2ml*6개입,3000.0,2328.0


In [15]:
# A-3. Review Density (리뷰 밀도)
# Review Count / Likes (likes 대비 리뷰 작성 비율)
products_df['review_density'] = products_df.apply(
    lambda x: x['review_count'] / x['likes'] if x['likes'] > 0 else 0,
    axis=1
)

print("Review Density 생성 완료")
print(f"평균 리뷰 밀도: {products_df['review_density'].mean():.4f}")
print(f"최대 리뷰 밀도: {products_df['review_density'].max():.4f}")
print("\n리뷰 밀도가 높은 제품 TOP 5")
print(products_df.nlargest(5, 'review_density')[['name', 'brand', 'likes', 'review_count', 'review_density']])

Review Density 생성 완료
평균 리뷰 밀도: 0.1655
최대 리뷰 밀도: 1.5455

리뷰 밀도가 높은 제품 TOP 5
                                  name   brand  likes  review_count  \
480     [03 펀치 오렌지] 코드글로컬러 프루티 볼륨 립글로스  코드글로컬러     22            34   
474        [02 애플팝] 코드글로컬러 프루티 볼륨 립글로스  코드글로컬러     60            81   
475   [01 크랜베리 에이드] 코드글로컬러 프루티 볼륨 립글로스  코드글로컬러     40            40   
296  VT 리들샷 100 페이셜 부스팅 퍼스트 앰플 2ml*6개입      VT   9999          9789   
293  VT 리들샷 300 페이셜 부스팅 퍼스트 앰플 2ml*6개입      VT   9999          9784   

     review_density  
480        1.545455  
474        1.350000  
475        1.000000  
296        0.978998  
293        0.978498  


## B. 리뷰 관점

In [16]:
# B-1. Review Length (리뷰 길이)
reviews_df['review_length'] = reviews_df['text'].fillna('').str.len()

print("[B-1] Review Length 생성 완료")
print(f"평균 리뷰 길이: {reviews_df['review_length'].mean():.2f}자")
print(f"중간값: {reviews_df['review_length'].median():.0f}자")
print(f"최대: {reviews_df['review_length'].max()}자")

# 리뷰 길이 구간별 분류
def classify_review_length(length):
    if length < 20:
        return 'Very Short'
    elif length < 50:
        return 'Short'
    elif length < 100:
        return 'Medium'
    else:
        return 'Long'

reviews_df['review_length_category'] = reviews_df['review_length'].apply(classify_review_length)
print("\n리뷰 길이 분포:")
print(reviews_df['review_length_category'].value_counts())

[B-1] Review Length 생성 완료
평균 리뷰 길이: 35.73자
중간값: 24자
최대: 996자

리뷰 길이 분포:
review_length_category
Short         178569
Very Short    142161
Medium         54375
Long           18134
Name: count, dtype: int64


## C. 유저 관점

In [17]:
# C-0. 재구매 명시 여부 (텍스트가 '재구매'로 시작)
reviews_df['is_reorder'] = reviews_df['text'].fillna('').str.strip().str.startswith('재구매')

print("is_reorder 생성 완료")
reorder_count = reviews_df['is_reorder'].sum()
print(f"\n'재구매'로 시작하는 리뷰: {reorder_count:,}건 ({reorder_count/len(reviews_df)*100:.2f}%)")

is_reorder 생성 완료

'재구매'로 시작하는 리뷰: 116,055건 (29.51%)


In [18]:
# C-1. User Activity Level (유저 활동 등급)
user_review_counts = reviews_df.groupby('user_id').size().reset_index(name='user_total_reviews_calc')

# BQ에서 이미 user_total_reviews가 있으면 drop 후 재병합
for col in ['user_total_reviews', 'user_activity_level']:
    if col in reviews_df.columns:
        reviews_df = reviews_df.drop(columns=[col])

reviews_df = reviews_df.merge(user_review_counts, on='user_id', how='left')
reviews_df = reviews_df.rename(columns={'user_total_reviews_calc': 'user_total_reviews'})

def classify_user_activity(count):
    if count <= 1:
        return 'Newbie'
    elif count <= 5:
        return 'Junior'
    elif count <= 20:
        return 'Regular'
    else:
        return 'VIP'

reviews_df['user_activity_level'] = reviews_df['user_total_reviews'].apply(classify_user_activity)

print("User Activity Level 생성 완료")
print("\n유저 등급 분포:")
print(reviews_df['user_activity_level'].value_counts())
print(f"\n고유 user_masked 수: {reviews_df['user_masked'].nunique():,}명")
print(f"\n고유 user 수: {reviews_df['user_id'].nunique():,}명")

User Activity Level 생성 완료

유저 등급 분포:
user_activity_level
VIP        272596
Regular     86585
Junior      29011
Newbie       5047
Name: count, dtype: int64

고유 user_masked 수: 26,800명

고유 user 수: 26,800명


In [19]:
# C-2. User Average Rating (재구매 고객의 평균 평점)

reorder_reviews_only = reviews_df[reviews_df['is_reorder'] == True].copy()

print("User Average Rating (재구매 고객 기준) 생성")
print(f"전체 리뷰: {len(reviews_df):,}건")
print(f"재구매 리뷰: {len(reorder_reviews_only):,}건")

if len(reorder_reviews_only) > 0:
    user_avg_rating_reorder = reorder_reviews_only.groupby('user_id')['rating'].mean().reset_index(name='user_avg_rating_reorder')
    
    # 전체 리뷰에 병합 (재구매 경험 없는 유저는 NaN)
    reviews_df = reviews_df.merge(user_avg_rating_reorder, on='user_id', how='left')
    
    # 평점 부여 성향 분류
    def classify_rating_tendency(avg_rating):
        if pd.isna(avg_rating):
            return 'No Reorder'
        elif avg_rating >= 4.8:
            return 'Always Positive'
        elif avg_rating >= 4.0:
            return 'Mostly Positive'
        elif avg_rating >= 3.0:
            return 'Balanced'
        else:
            return 'Critical'
    
    reviews_df['user_rating_tendency'] = reviews_df['user_avg_rating_reorder'].apply(classify_rating_tendency)
    
    print("\nUser Average Rating 생성 완료")
    print(f"재구매 경험 있는 유저: {user_avg_rating_reorder['user_id'].nunique():,}명")
    print(f"재구매 리뷰 평균 평점: {user_avg_rating_reorder['user_avg_rating_reorder'].mean():.3f}")
    
    print("\n유저 평점 성향 분포:")
    print(reviews_df['user_rating_tendency'].value_counts())
    
else:
    reviews_df['user_avg_rating_reorder'] = np.nan
    reviews_df['user_rating_tendency'] = 'No Reorder'
    print("\n재구매 리뷰가 없습니다.")

User Average Rating (재구매 고객 기준) 생성
전체 리뷰: 393,239건
재구매 리뷰: 116,055건

User Average Rating 생성 완료
재구매 경험 있는 유저: 14,806명
재구매 리뷰 평균 평점: 4.824

유저 평점 성향 분포:
user_rating_tendency
Always Positive    264018
Mostly Positive     77725
No Reorder          44668
Balanced             5781
Critical             1047
Name: count, dtype: int64


In [20]:
# C-3. Repurchase Indicator
# is_reorder=True인 리뷰들만 기준으로 같은 브랜드/카테고리 재구매 확인

# 고유 식별자 추가 (원본 인덱스 보존)
reviews_df['_temp_idx'] = reviews_df.index

# 제품 정보 병합 (brand, category 정보 가져오기)
reviews_with_product = reviews_df.merge(
    products_df[['product_code', 'brand', 'category_2']], 
    on='product_code', 
    how='left'
)

# 날짜 변환
reviews_with_product['date'] = pd.to_datetime(reviews_with_product['date'])

# 유저별로 정렬
reviews_with_product = reviews_with_product.sort_values(['user_id', 'date'])

# is_reorder=True인 리뷰들만 필터링
reorder_reviews = reviews_with_product[reviews_with_product['is_reorder'] == True].copy()

print("Repurchase Indicator 생성 완료")
print(f"'재구매' 명시 리뷰: {len(reorder_reviews):,}건")

# 같은 브랜드 재구매 여부
if len(reorder_reviews) > 0:
    reorder_reviews['is_brand_repurchase'] = (
        reorder_reviews.groupby('user_id')['brand']
        .transform(lambda x: x.duplicated(keep=False))
    ).astype(int)
    
    # 같은 카테고리 재구매 여부
    reorder_reviews['is_category_repurchase'] = (
        reorder_reviews.groupby('user_id')['category_2']
        .transform(lambda x: x.duplicated(keep=False))
    ).astype(int)
    
    # 원본 reviews_df에 병합 (기본값은 0)
    reviews_df['is_brand_repurchase'] = 0
    reviews_df['is_category_repurchase'] = 0
    
    # _temp_idx를 기준으로 매칭하여 업데이트
    temp_mapping = reorder_reviews.set_index('_temp_idx')[['is_brand_repurchase', 'is_category_repurchase']]
    reviews_df.loc[temp_mapping.index, 'is_brand_repurchase'] = temp_mapping['is_brand_repurchase'].values
    reviews_df.loc[temp_mapping.index, 'is_category_repurchase'] = temp_mapping['is_category_repurchase'].values
    
    print(f"\n'재구매' 리뷰 중:")
    print(f"  - 같은 브랜드 재구매: {reviews_df['is_brand_repurchase'].sum():,}건")
    print(f"  - 같은 카테고리 재구매: {reviews_df['is_category_repurchase'].sum():,}건")
else:
    reviews_df['is_brand_repurchase'] = 0
    reviews_df['is_category_repurchase'] = 0
    print("\n'재구매' 명시 리뷰가 없습니다.")

# 임시 인덱스 컬럼 제거
reviews_df = reviews_df.drop(columns=['_temp_idx'])

Repurchase Indicator 생성 완료
'재구매' 명시 리뷰: 116,055건

'재구매' 리뷰 중:
  - 같은 브랜드 재구매: 92,721건
  - 같은 카테고리 재구매: 103,669건


## D. 시계열(Time) 관점

In [21]:
# D-1. Seasonality (계절성) - 년도별 분석
reviews_df['date'] = pd.to_datetime(reviews_df['date'])
reviews_df['year'] = reviews_df['date'].dt.year
reviews_df['month'] = reviews_df['date'].dt.month
reviews_df['day_of_week'] = reviews_df['date'].dt.dayofweek  # 0=월요일, 6=일요일
reviews_df['day_name'] = reviews_df['date'].dt.day_name()

# 계절 분류
def classify_season(month):
    if month in [3, 4, 5]:
        return 'Spring'
    elif month in [6, 7, 8]:
        return 'Summer'
    elif month in [9, 10, 11]:
        return 'Fall'
    else:
        return 'Winter'

reviews_df['season'] = reviews_df['month'].apply(classify_season)

print("Seasonality 생성 완료")
print("="*80)

# 1. 년도별 리뷰 수
print("\n[1] 년도별 리뷰 수")
print("-"*80)
year_counts = reviews_df['year'].value_counts().sort_index()
print(year_counts)

# 2. 년도별 월별 리뷰 수
print("\n[2] 년도별 월별 리뷰 수")
print("-"*80)
year_month_counts = reviews_df.groupby(['year', 'month']).size().reset_index(name='count')
for year in sorted(reviews_df['year'].unique()):
    year_data = year_month_counts[year_month_counts['year'] == year]
    if len(year_data) > 0:
        print(f"\n{year}년:")
        for _, row in year_data.iterrows():
            print(f"  {int(row['month']):2d}월: {int(row['count']):,}건")

# 3. 년도별 계절별 리뷰 수
print("\n[3] 년도별 계절별 리뷰 수")
print("-"*80)
year_season_counts = reviews_df.groupby(['year', 'season']).size().reset_index(name='count')
for year in sorted(reviews_df['year'].unique()):
    year_data = year_season_counts[year_season_counts['year'] == year]
    if len(year_data) > 0:
        print(f"\n{year}년:")
        season_order = ['Spring', 'Summer', 'Fall', 'Winter']
        for season in season_order:
            season_data = year_data[year_data['season'] == season]
            if len(season_data) > 0:
                count = int(season_data['count'].values[0])
                print(f"  {season:8s}: {count:,}건")

# 4. 전체 요일별 분포
print("\n[4] 요일별 리뷰 수 (전체)")
print("-"*80)
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
day_counts = reviews_df['day_name'].value_counts()
for day in day_order:
    if day in day_counts.index:
        print(f"{day:10s}: {day_counts[day]:,}건")

Seasonality 생성 완료

[1] 년도별 리뷰 수
--------------------------------------------------------------------------------
year
2021        24
2022         2
2023        18
2024     96094
2025    233647
2026     63454
Name: count, dtype: int64

[2] 년도별 월별 리뷰 수
--------------------------------------------------------------------------------

2021년:
   1월: 4건
   4월: 2건
   6월: 6건
   8월: 2건
   9월: 2건
  10월: 2건
  12월: 6건

2022년:
   1월: 2건

2023년:
   3월: 2건
   9월: 2건
  12월: 14건

2024년:
   1월: 1,946건
   2월: 3,673건
   3월: 5,202건
   4월: 5,188건
   5월: 7,002건
   6월: 8,563건
   7월: 10,811건
   8월: 9,794건
   9월: 8,295건
  10월: 9,803건
  11월: 10,760건
  12월: 15,057건

2025년:
   1월: 18,242건
   2월: 17,861건
   3월: 23,376건
   4월: 22,677건
   5월: 21,598건
   6월: 16,987건
   7월: 19,120건
   8월: 18,900건
   9월: 18,554건
  10월: 16,749건
  11월: 18,831건
  12월: 20,752건

2026년:
   1월: 22,421건
   2월: 30,868건
   3월: 10,165건

[3] 년도별 계절별 리뷰 수
--------------------------------------------------------------------------------

2021년:
  Spri

In [22]:
# D-2. Days Since First Review (출시 후 경과일)
# 제품별 첫 리뷰 날짜
first_review_dates = reviews_df.groupby('product_code')['date'].min().reset_index(name='first_review_date')
reviews_df = reviews_df.merge(first_review_dates, on='product_code', how='left')

# 경과일 계산
reviews_df['days_since_first_review'] = (
    reviews_df['date'] - reviews_df['first_review_date']
).dt.days

# 신상품 여부 (첫 리뷰 후 30일 이내)
reviews_df['is_new_product'] = (reviews_df['days_since_first_review'] <= 30).astype(int)

print("Days Since First Review 생성 완료")
print(f"평균 경과일: {reviews_df['days_since_first_review'].mean():.1f}일")
print(f"최대 경과일: {reviews_df['days_since_first_review'].max()}일")
print(f"\n신상품(30일 이내) 리뷰: {reviews_df['is_new_product'].sum():,}건 ({reviews_df['is_new_product'].sum()/len(reviews_df)*100:.2f}%)")

Days Since First Review 생성 완료
평균 경과일: 295.7일
최대 경과일: 1874일

신상품(30일 이내) 리뷰: 43,127건 (10.97%)


## E. 프로모션 관점

In [23]:
# 프로모션 데이터 (BigQuery)
promo_df = query_to_df("""
SELECT
    promotion_id,
    description,
    brand_id,
    event_type,
    start_date,
    end_date
FROM daiso.promotions
""")
promo_df['start_date'] = pd.to_datetime(promo_df['start_date'])
promo_df['end_date']   = pd.to_datetime(promo_df['end_date'])

# brands 이름 매핑 (brand_id -> brand name) + date 호환성
_brands = query_to_df("SELECT brand_id, name AS brand FROM daiso.brands")
promo_df = promo_df.merge(_brands, on='brand_id', how='left')
promo_df['brand'] = promo_df['brand'].fillna('-')
promo_df['date']  = promo_df['start_date']

print(f"프로모션 데이터: {len(promo_df):,}건")

프로모션 데이터: 16건


In [24]:
# E-3. 프로모션 기간 여부 확인
# 리뷰 날짜가 프로모션 날짜와 일치하거나 직후 7일 이내인 경우
from datetime import timedelta

# promo_type_category가 없으면 생성
if 'promo_type_category' not in promo_df.columns:
    def categorize_promo_type(event_type, description):
        event_type = str(event_type).lower()
        description = str(description).lower()
        if '리뷰' in event_type or 'review' in event_type:
            return '리뷰이벤트'
        elif '할인' in description or 'sale' in description:
            return '할인'
        elif '증정' in description or 'gift' in description:
            return '증정'
        elif '구매' in event_type:
            return '구매이벤트'
        else:
            return '기타'
    promo_df['promo_type_category'] = promo_df.apply(
        lambda x: categorize_promo_type(x['event_type'], x['description']), axis=1
    )

# [수정] 중복 제거 후 병합 (product_code 기준으로 첫 번째만 유지)
products_brand_unique = products_df[['product_code', 'brand']].drop_duplicates(subset='product_code', keep='first')
reviews_with_brand = reviews_df.merge(
    products_brand_unique,
    on='product_code',
    how='left'
)

print(f"현재 reviews_df: {len(reviews_df):,}건")
print(f"병합 후 reviews_with_brand: {len(reviews_with_brand):,}건")
if len(reviews_df) != len(reviews_with_brand):
    print("경고: 병합 후 행 수가 다릅니다!")
else:
    print("병합 성공: 행 수 일치")

# [수정] 벡터화 방식으로 프로모션 매칭 (성능 개선)
# 1. 프로모션 날짜 범위 생성 (date ~ date+7일)
promo_expanded = []
for _, promo in promo_df.iterrows():
    for day_offset in range(8):  # 0~7일
        promo_expanded.append({
            'promo_date': promo['date'] + timedelta(days=day_offset),
            'promo_brand': promo['brand'],
            'promo_type': promo['promo_type_category']
        })
promo_expanded_df = pd.DataFrame(promo_expanded)

# 2. 리뷰 날짜를 date로 변환
reviews_with_brand['date'] = pd.to_datetime(reviews_with_brand['date'])

# 3. 날짜 기준 병합 (브랜드별 또는 전체 프로모션)
# 전체 프로모션 ('-') 먼저 매칭
global_promos = promo_expanded_df[promo_expanded_df['promo_brand'] == '-'][['promo_date', 'promo_type']].drop_duplicates()
global_promos = global_promos.rename(columns={'promo_date': 'date', 'promo_type': 'global_promo_type'})

reviews_with_brand = reviews_with_brand.merge(
    global_promos,
    on='date',
    how='left'
)

# 브랜드별 프로모션 매칭
brand_promos = promo_expanded_df[promo_expanded_df['promo_brand'] != '-'].copy()
brand_promos = brand_promos.rename(columns={'promo_date': 'date', 'promo_brand': 'brand', 'promo_type': 'brand_promo_type'})
brand_promos = brand_promos.drop_duplicates(subset=['date', 'brand'])

reviews_with_brand = reviews_with_brand.merge(
    brand_promos,
    on=['date', 'brand'],
    how='left'
)

# 4. 최종 프로모션 여부 결정 (브랜드 프로모션 > 전체 프로모션)
reviews_with_brand['promo_type_category'] = reviews_with_brand['brand_promo_type'].fillna(
    reviews_with_brand['global_promo_type']
).fillna('프로모션 기간 아님')

reviews_with_brand['is_during_promo'] = reviews_with_brand['promo_type_category'] != '프로모션 기간 아님'

# 5. 원본 reviews_df에 결과 반영
reviews_df['is_during_promo'] = reviews_with_brand['is_during_promo'].values
reviews_df['promo_type_category'] = reviews_with_brand['promo_type_category'].values

print(f"\n완료! (벡터화 방식으로 처리)")

현재 reviews_df: 393,239건
병합 후 reviews_with_brand: 393,239건
병합 성공: 행 수 일치

완료! (벡터화 방식으로 처리)


In [25]:
# 결과 확인
promo_reviews = reviews_df['is_during_promo'].sum()
print("[E-4] 프로모션 파생변수 결과")
print("="*80)
print(f"\n프로모션 기간 중/직후 리뷰: {promo_reviews:,}건 ({promo_reviews/len(reviews_df)*100:.2f}%)")
print(f"\n프로모션 유형별 리뷰 수:")
print(reviews_df['promo_type_category'].value_counts())
print(f"\n리뷰이벤트 기간 리뷰 비율:")
review_event_count = (reviews_df['promo_type_category'] == '리뷰이벤트').sum()
print(f"  {review_event_count:,}건 ({review_event_count/len(reviews_df)*100:.2f}%)")


[E-4] 프로모션 파생변수 결과

프로모션 기간 중/직후 리뷰: 4,266건 (1.08%)

프로모션 유형별 리뷰 수:
promo_type_category
프로모션 기간 아님    388973
리뷰이벤트           4120
구매이벤트            146
Name: count, dtype: int64

리뷰이벤트 기간 리뷰 비율:
  4,120건 (1.05%)
